# RL Vectorizer — Colab SFT Training

This notebook runs SFT training with intermediate instructions on ResPlan data.
It uses T4 GPU (free) to train Qwen2.5-VL-3B with LoRA.

In [ ]:
# @title 1. Clone repo & install dependencies
import os
GITHUB_URL = "https://github.com/YOUR_USERNAME/rl_vectorizer.git"  # ← 改成你的仓库地址

!git clone $GITHUB_URL
%cd rl_vectorizer

# Install dependencies
!pip install -q torch transformers peft accelerate bitsandbytes datasets
!pip install -q lxml cairosvg pillow scikit-image opencv-python tensorboard
!pip install -q hydra-core omegaconf shapely

In [ ]:
# @title 2. Download & prepare ResPlan data
import os, urllib.request, zipfile, pickle

DATA_DIR = "data/resplan"
os.makedirs(DATA_DIR, exist_ok=True)

ZIP_PATH = os.path.join(DATA_DIR, "ResPlan.zip")
if not os.path.exists(ZIP_PATH):
    print("Downloading ResPlan.zip (96MB)...")
    urllib.request.urlretrieve(
        "https://github.com/m-agour/ResPlan/releases/download/1.0.0/ResPlan.zip",
        ZIP_PATH)
    print("Download complete!")

# Extract pkl
PKL_PATH = os.path.join(DATA_DIR, "ResPlan.pkl")
if not os.path.exists(PKL_PATH):
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    print("Extracted!")

# Convert pkl → SVG + PNG
if not os.path.exists(os.path.join(DATA_DIR, "svgs")):
    !python convert_resplan.py

# Prepare SFT data
if not os.path.exists(os.path.join(DATA_DIR, "sft_train.jsonl")):
    !python scripts/prepare_sft_data.py

print("Data ready!")
!ls -lh {DATA_DIR}/sft_train.jsonl

In [ ]:
# @title 3. Run SFT Training
import os
# 限制样本数（第一次运行设小一点，成功后改 None 跑全部）
MAX_SAMPLES = 200  # or None for full 17k
BATCH_SIZE = 1
EPOCHS = 3
LR = 1e-4

!python scripts/train_sft.py \
    --max-samples {MAX_SAMPLES} \
    --batch-size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --lr {LR} \
    --save-dir checkpoints/sft \
    --log-interval 5

In [ ]:
# @title 4. Inference & Save Results
import torch
from PIL import Image
from transformers import AutoProcessor
from peft import PeftModel
from transformers import Qwen2_5_VLForConditionalGeneration

# Load base model + LoRA
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, "checkpoints/sft/final")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", use_fast=False)

# Test on a sample image
img = Image.open("data/resplan/bitmaps/resplan_00013.png").convert("RGB")
messages = [
    {"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": "Convert this floor plan to SVG."},
    ]}
]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[img], return_tensors="pt").to("cuda")

with torch.no_grad():
    gen = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    response = processor.decode(gen[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(response[:1000])

# Save
with open("colab_output.txt", "w") as f:
    f.write(response)
from google.colab import files
files.download("colab_output.txt")